In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col

In [0]:
df = spark.sql("""
    SELECT *
    FROM mis_automation.silver_credit_card.vw_credit_card_history
    WHERE snapshot_date >= APPROVEDATE
      AND snapshot_date < add_months(APPROVEDATE, 24)
      AND APPROVEDATE > '2022-06-30'
      AND APPROVEDATE < '2025-07-01'
""")

# Exclusion 9 needs a per-account count of OPEN snapshots first
w = Window.partitionBy("CREDITCARDNO")  # replace with your actual account/customer key
df = df.withColumn(
    "open_count",
    F.sum(F.when(F.col("Current_status") == "1.OPEN", 1).otherwise(0)).over(w)
)

exclude_cond = (
    (F.upper(F.trim(F.col("CLASSCODE"))) == "SME") |                                                   # Exclusion 1
    (~F.upper(F.trim(F.col("CARD_BUSINESS_SEGMENT"))).isin("CARDS")) |                                 # Exclusion 2
    (F.col("VIP") == 1) | (F.upper(F.trim(F.col("RULING_FAMILY_FLAG"))) == 'Y') |                      # Exclusion 3
 #   (F.col("AECB_DATA").isNull()) |                                                                   # Exclusion 4
    (F.upper(F.trim(F.col("EMPLOYMENT_CATEGORY"))) != "Salaried".upper()) |                              # Exclusion 6
    (F.col("PRIMARYCARD") == 0) |                                                      # Exclusion 7
    (F.upper(F.trim(F.col("CLASSCODE"))) == "STAFF") |                                                  # Exclusion 8
    (F.col("open_count") < 3)                                                          # Exclusion 9
)

df_clean = df.filter(~exclude_cond).drop("open_count")

cols = ['snapshot_date','CRNNO','CREDITCARDNO','CARDCREATEDATE','APPROVEDATE','DELINQUENCYBUCKET']

print(df_clean.columns)

In [0]:
tmp = df.filter(F.col("CREDITCARDNO") == "5239267539247004").toPandas()
tmp[["snapshot_date","CRNNO","CREDITCARDNO","CARDPRODUCT","EMPLOYMENTTYPE","EMPLOYMENT_CATEGORY","APPROVEDATE","LOS","open_count","Current_status"]].sort_values("snapshot_date")

In [0]:
df_clean.describe().show()
df_pd = df_clean.toPandas()

In [0]:
from pyspark.sql.functions import col
# df.filter(~col("APPROVEDDT").eqNullSafe(col("APPROVEDATE"))).count()

df.groupBy("EMPLOYMENT_CATEGORY").count().orderBy("count", ascending=False).show()

In [0]:
pandas_df = df.sample(withReplacement=False, fraction=0.1, seed=42).limit(1000).toPandas()
pandas_df.head(12)

In [0]:
cols = ['snapshot_date','CRNNO','CREDITCARDNO','CARDCREATEDATE','APPROVEDATE','DELINQUENCYBUCKET']

df_pd.groupby(['CREDITCARDNO'])["Current_status"].value_counts().unstack().reset_index()['1.OPEN'].value_counts()

In [0]:
df_pd.loc[df_pd['CREDITCARDNO']=="5239267539247004"]

In [0]:
data = df_pd[cols]

In [0]:
data.describe()

In [0]:
data[cols].groupby(['CREDITCARDNO','APPROVEDATE']).agg(los=('snapshot_date',pd.Series.nunique)).reset_index()

In [0]:
df[cols].loc[df.CREDITCARDNO=="1531000000370110"].sort_values("snapshot_date")#"1531000004671703"

In [0]:
df_del = df_pd[["CREDITCARDNO","snapshot_date",'DELINQUENCYBUCKET','APPROVEDATE']].groupby(["CREDITCARDNO"]).agg(max_delinquencybucket=('DELINQUENCYBUCKET',max),min_approvedate=('APPROVEDATE',min),max_snapshotdate=('snapshot_date',max)).reset_index()

In [0]:
df_del.head()

In [0]:
df_del["target_12m_date"] = df_del['min_approvedate']+pd.DateOffset(months=12)
df_del.head()


In [0]:
df_del.loc[df_del.target_12m_date < df_del.max_snapshotdate]

In [0]:
import matplotlib.pyplot as plt
df_del['max_delinquencybucket'] = pd.to_numeric(df_del['max_delinquencybucket'], errors='coerce').fillna(-1).astype(int)
df_del['max_delinquencybucket'].value_counts().sort_index()
df_del['max_delinquencybucket'].loc[df_del['max_delinquencybucket']>0].value_counts().sort_index().plot(kind="bar",figsize=(8,4))
plt.title("Distibution of Deliquency Buckets")
plt.xlabel("Bucket")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [0]:
!pip install --upgrade optuna statsmodels scikit-learn gputil dvc apache-airflow lightgbm xgboost catboost h2o python-tds



In [0]:
dff = prepare_data(data, id_col="CREDITCARDNO", snap_col="snapshot_date",
                  bucket_col="DELINQUENCYBUCKET", approve_col="APPROVEDATE")
dff.head()

In [0]:
transition_rate, transition_counts = build_roll_rate_matrix(dff, id_col="CREDITCARDNO", snap_col="snapshot_date", bucket_col="DELINQUENCYBUCKET", n_periods=1)
transition_rate.head()

In [0]:
transition_rate, transition_counts = build_roll_rate_matrix(dff, id_col="CREDITCARDNO", snap_col="snapshot_date", bucket_col="DELINQUENCYBUCKET", n_periods=1)
transition_rate.head()

In [0]:
n_periods = 1

# Professional style
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(8, 6))

im = ax.imshow(transition_rate.values, cmap="Reds", vmin=0, vmax=100, aspect="auto")

# Axis ticks and labels
ax.set_xticks(range(len(transition_rate.columns)))
ax.set_yticks(range(len(transition_rate.index)))
ax.set_xticklabels([f"Bucket {c}" for c in transition_rate.columns], fontsize=10, fontweight='medium', rotation=45, ha='right')
ax.set_yticklabels([f"Bucket {r}" for r in transition_rate.index], fontsize=10, fontweight='medium')

# Move x-tick labels to bottom and ensure visibility
ax.tick_params(axis='x', bottom=True, top=False, labelbottom=True, labeltop=False, pad=8)
ax.tick_params(axis='y', pad=5)

# Axis labels
ax.set_xlabel(f"Bucket after {n_periods} month(s)", fontsize=12, fontweight='bold', labelpad=12)
ax.set_ylabel("Current Bucket", fontsize=12, fontweight='bold', labelpad=10)

# Title
ax.set_title(f"Roll Rate Matrix ({n_periods}-Month Transition, %)", fontsize=14, fontweight='bold', pad=15)

# Annotate cells with values
for i in range(transition_rate.shape[0]):
    for j in range(transition_rate.shape[1]):
        
        value = transition_rate.values[i, j]
        # Use white text on dark cells for readability
        text_color = "white" if value > 60 else "black"
        ax.text(j, i, f"{value:.1f}",
                ha="center", va="center", fontsize=10, fontweight='semibold', color=text_color)

# Colorbar
cbar = fig.colorbar(im, ax=ax, label="Transition Probability (%)", shrink=0.85)
cbar.ax.tick_params(labelsize=9)
cbar.set_label("Transition Probability (%)", fontsize=11)

# Remove gridlines from imshow
ax.grid(False)

plt.tight_layout()
plt.show()

In [0]:
transition_counts

In [0]:
transition_counts

In [0]:
transition_counts

In [0]:
# rate, total = eventual_roll_forward_rate(dff, id_col="CREDITCARDNO", snap_col="snapshot_date",
#                                 bucket_col="DELINQUENCYBUCKET",
#                                 from_bucket=2, to_bucket_min=4, within_months=6)
# rate.head(4)
# total.head(4)
account_summary, bad_definitions = define_bad_flags(dff,id_col="CREDITCARDNO", bucket_col="DELINQUENCYBUCKET", mob_col="MOB", bad_definitions=None)
account_summary.head()
# 
                        

In [0]:
summary_df, overlap_df = compare_bad_definitions(account_summary, bad_definitions)
overlap_df

In [0]:
account_summary.head()

In [0]:
# Filter out vintages greater than 2024-07 to keep only mature cohorts
dff_mature = dff[dff["_approve_month"] < pd.Timestamp("2024-07-01")]
account_summary_mature = account_summary[account_summary["approve_month"] < pd.Timestamp("2024-07-01")]

vintage_cum_bad_rate = vintage_cumulative_bad_rate(dff_mature, account_summary_mature, id_col="CREDITCARDNO",
                                 bucket_col="DELINQUENCYBUCKET", min_bucket=3,
                                 mob_col="MOB", max_mob=24)
vintage_cum_bad_rate_ = vintage_cum_bad_rate.dropna()

In [0]:
vintage_cum_bad_rate_.loc['2022-08']

In [0]:
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(12, 7))

cmap = plt.cm.viridis
colors = [cmap(i / max(len(vintage_cum_bad_rate_.index) - 1, 1)) for i in range(len(vintage_cum_bad_rate_.index))]

for idx, vintage in enumerate(vintage_cum_bad_rate_.index):
    series = vintage_cum_bad_rate_.loc[vintage].dropna()
    ax.plot(series.index, series.values, marker="o", markersize=4, linewidth=1.8,
            color=colors[idx], label=vintage)

ax.set_xlabel("Months on Book (MOB)", fontsize=12, fontweight='bold', labelpad=10)
ax.set_ylabel("Cumulative Bad Rate (%)", fontsize=12, fontweight='bold', labelpad=10)
ax.set_title("Vintage Cumulative Bad Rate Analysis", fontsize=14, fontweight='bold', pad=15)
ax.legend(title="Vintage", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8,
          title_fontsize=10, frameon=True, framealpha=0.9)
ax.grid(alpha=0.3, linestyle='--')
ax.tick_params(axis='both', labelsize=10)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

In [0]:
mature_mob, avg_curve, marginal = find_maturity_point(vintage_cum_bad_rate, marginal_threshold=0.3)
mature_mob, avg_curve, marginal

In [0]:
"""
Roll Rate & Vintage Analysis — 用于确定信用卡 Bad 定义 (Target Variable)
=========================================================================

场景：你有逐月快照数据，字段包括
    CREDITCARDNO      : 卡号（如果你的原始表字段是 CREDTICARDNO，直接改 id_col 参数即可）
    SNAPSHOT_DATE      : 月度快照日期
    DELINQUENCYBUCKET  : 0=正常, 1=1-29天逾期, 2=30-59天逾期, 3=60-89天逾期, 4=90+天逾期
                         （按你实际业务口径调整 bucket 数值含义）
    APPROVEDATE     : 开卡/激活日期

目标：通过
    1) Roll Rate 分析 —— 一旦进入某个逾期阶段，未来滚到更差阶段的概率有多大
    2) Vintage 分析   —— 不同开卡批次(Vintage)的累计坏账率随账龄(MOB)如何演化，
                         曲线什么时候开始走平（即"成熟点"）
来帮你在多个候选 Bad 定义（如 Ever60DPD_12M / Ever30DPD_6M / Ever90DPD_12M）之间做选择。
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =============================================================================
# 0. 数据准备：计算 MOB (Months on Book)
# =============================================================================

def prepare_data(df, id_col="CREDITCARDNO", snap_col="snapshot_date",
                  bucket_col="DELINQUENCYBUCKET", approve_col="APPROVEDATE"):
    """
    清洗 + 计算 MOB。返回处理后的 long-format dataframe，
    新增列: _snap_month, _approve_month, MOB
    """
    df = df.copy()
    df[snap_col] = pd.to_datetime(df[snap_col])
    df[approve_col] = pd.to_datetime(df[approve_col])

    # 统一取月初，避免同月内日期不一致导致 MOB 计算错位
    df["_snap_month"] = df[snap_col].values.astype("datetime64[M]")
    df["_approve_month"] = df[approve_col].values.astype("datetime64[M]")

    df["MOB"] = (
        (df["_snap_month"].dt.year - df["_approve_month"].dt.year) * 12
        + (df["_snap_month"].dt.month - df["_approve_month"].dt.month)
    )

    before = len(df)
    df = df[df["MOB"] >= 0].copy()  # 剔除快照早于开户日的脏数据
    dropped = before - len(df)
    if dropped > 0:
        print(f"[prepare_data] 剔除了 {dropped} 条 MOB<0 的异常记录（快照早于开户日）")

    df = df.sort_values([id_col, snap_col]).reset_index(drop=True)
    return df


# =============================================================================
# 1. Roll Rate 分析
# =============================================================================

def build_roll_rate_matrix(df, id_col="CREDITCARDNO", snap_col="snapshot_date",
                            bucket_col="DELINQUENCYBUCKET", n_periods=1):
    """
    n_periods 个月的滚动迁移矩阵：
    Row = 当前 Bucket, Column = n_periods 个月后的 Bucket, Value = 迁移概率(%)
    只统计真正间隔恰好 n_periods 个月、且账户在两个时点都有记录的数据对，
    避免因为账户数据缺月造成的虚假跳跃。
    """
    df = df.sort_values([id_col, snap_col]).copy()
    df["_period"] = df[snap_col].dt.to_period("M")

    df["_next_bucket"] = df.groupby(id_col)[bucket_col].shift(-1)
    df["_next_period"] = df.groupby(id_col)["_period"].shift(-1)

    valid = df.dropna(subset=["_next_bucket", "_next_period"]).copy()
    gap = (valid["_next_period"] - valid["_period"]).apply(lambda x: x.n)
    valid = valid[gap == n_periods]

    transition_counts = pd.crosstab(valid[bucket_col], valid["_next_bucket"])
    transition_rate = transition_counts.div(transition_counts.sum(axis=1), axis=0) * 100
    return transition_rate.round(2), transition_counts


def plot_roll_rate_heatmap(transition_rate, n_periods=1, save_path=None):
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(transition_rate.values, cmap="Reds", vmin=0, vmax=100)
    ax.set_xticks(range(len(transition_rate.columns)))
    ax.set_yticks(range(len(transition_rate.index)))
    ax.set_xticklabels([f"Bucket {c}" for c in transition_rate.columns])
    ax.set_yticklabels([f"Bucket {r}" for r in transition_rate.index])
    ax.set_xlabel(f"Bucket after {n_periods} month(s)")
    ax.set_ylabel("Current Bucket")
    ax.set_title(f"Roll Rate Matrix ({n_periods}-Month Transition, %)")
    for i in range(transition_rate.shape[0]):
        for j in range(transition_rate.shape[1]):
            ax.text(j, i, f"{transition_rate.values[i, j]:.1f}",
                    ha="center", va="center", fontsize=9)
    fig.colorbar(im, ax=ax, label="Transition Probability (%)")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
    return fig


def eventual_roll_forward_rate(df, id_col="CREDITCARDNO", snap_col="snapshot_date",
                                bucket_col="DELINQUENCYBUCKET",
                                from_bucket=2, to_bucket_min=4, within_months=6):
    """
    核心指标（用来判断某个 DPD 阶段是不是"回不去了"）：
    对所有曾处于 from_bucket 状态的账户-月份观察点，
    计算未来 within_months 个月内"最终"是否曾经达到 >= to_bucket_min 的比例。

    例：from_bucket=2 (30-59dpd), to_bucket_min=4 (90+dpd), within_months=6
    -> "一旦进入 30+ 逾期，未来 6 个月内滚到 90+ 逾期的概率有多大？"
    如果这个比例很高（比如 >80%），说明 30dpd 已经是一个很强的 bad 信号，
    可以考虑把 bad 定义提前到 30dpd，而不必等到 90dpd。

    注意：这里用逐账户循环实现，逻辑清晰但在千万级数据量下会慢，
    大数据量场景建议改用 groupby + rolling window 的向量化写法。
    """
    df = df.sort_values([id_col, snap_col]).copy()
    hits, total = 0, 0
    for _, g in df.groupby(id_col):
        g = g.reset_index(drop=True)
        at_from = g[g[bucket_col] == from_bucket]
        for _, row in at_from.iterrows():
            future = g[(g[snap_col] > row[snap_col]) &
                       (g[snap_col] <= row[snap_col] + pd.DateOffset(months=within_months))]
            if len(future) == 0:
                continue
            total += 1
            if (future[bucket_col] >= to_bucket_min).any():
                hits += 1
    rate = (hits / total * 100) if total > 0 else np.nan
    return rate, total


# =============================================================================
# 2. Bad 定义标注（account-level）
# =============================================================================

def define_bad_flags(df, id_col="CREDITCARDNO", bucket_col="DELINQUENCYBUCKET",
                      mob_col="MOB", bad_definitions=None):
    """
    bad_definitions 示例:
        {
            "Ever60DPD_12M": {"min_bucket": 3, "window_months": 12},
            "Ever30DPD_6M":  {"min_bucket": 2, "window_months": 6},
            "Ever90DPD_12M": {"min_bucket": 4, "window_months": 12},
        }
    (min_bucket 按你实际的 DELINQUENCYBUCKET 编码调整；本例假设
     0=正常 1=1-29dpd 2=30-59dpd 3=60-89dpd 4=90+dpd)

    返回 account-level dataframe：每个账户一行，
    标注是否 mature（观察窗口是否已跑满）、是否触发 bad。
    """
    if bad_definitions is None:
        bad_definitions = {
            "Ever60DPD_12M": {"min_bucket": 3, "window_months": 12},
            "Ever30DPD_6M":  {"min_bucket": 2, "window_months": 6},
            "Ever90DPD_12M": {"min_bucket": 4, "window_months": 12},
        }

    account_summary = df.groupby(id_col).agg(
        max_mob_observed=("MOB", "max"),
        approve_month=("_approve_month", "first"),
    ).reset_index()

    for name, cfg in bad_definitions.items():
        window = cfg["window_months"]
        min_bucket = cfg["min_bucket"]

        account_summary[f"{name}_mature"] = account_summary["max_mob_observed"] >= window

        window_df = df[df[mob_col] <= window]
        bad_flags = (
            window_df.groupby(id_col)[bucket_col]
            .apply(lambda s: bool((s >= min_bucket).any()))
            .rename(f"{name}_bad")
        )
        account_summary = account_summary.merge(bad_flags, on=id_col, how="left")
        account_summary[f"{name}_bad"] = account_summary[f"{name}_bad"].fillna(False)

    return account_summary, bad_definitions


def compare_bad_definitions(account_summary, bad_definitions):
    """
    对比不同 Bad 定义：
      - 各定义在"成熟样本"口径下的 Bad Rate
      - 两两定义之间的一致性(Agreement%) 和相关性(correlation)
    """
    names = list(bad_definitions.keys())
    rows = []
    for name in names:
        mature = account_summary[f"{name}_mature"]
        pop = account_summary[mature]
        bad_rate = pop[f"{name}_bad"].mean() * 100 if len(pop) else np.nan
        rows.append({
            "definition": name,
            "mature_sample_size": int(mature.sum()),
            "bad_rate_%": round(bad_rate, 2),
        })
    summary_df = pd.DataFrame(rows)

    overlap_rows = []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            n1, n2 = names[i], names[j]
            both_mature = account_summary[f"{n1}_mature"] & account_summary[f"{n2}_mature"]
            sub = account_summary[both_mature]
            if len(sub) == 0:
                continue
            agree = (sub[f"{n1}_bad"] == sub[f"{n2}_bad"]).mean() * 100
            corr = sub[f"{n1}_bad"].astype(int).corr(sub[f"{n2}_bad"].astype(int))
            overlap_rows.append({
                "pair": f"{n1} vs {n2}",
                "sample_size": len(sub),
                "agreement_%": round(agree, 2),
                "correlation": round(corr, 3),
            })
    overlap_df = pd.DataFrame(overlap_rows)
    return summary_df, overlap_df


# =============================================================================
# 3. Vintage 分析
# =============================================================================

def vintage_cumulative_bad_rate(df, account_summary, id_col="CREDITCARDNO",
                                 bucket_col="DELINQUENCYBUCKET", min_bucket=3,
                                 mob_col="MOB", max_mob=24):
    """
    对每个开户月份的 Vintage Cohort，计算累计 Bad Rate 随 MOB 增长的曲线。
    返回 pivot table: index=vintage(开户月, 如 '2025-01'), columns=MOB, values=累计Bad率(%)
    """
    merged = df.merge(
        account_summary[[id_col, "approve_month"]], on=id_col, how="left"
    )
    merged["vintage"] = merged["approve_month"].dt.to_period("M").astype(str)

    merged = merged.sort_values([id_col, mob_col])
    merged["_is_bad_event"] = merged[bucket_col] >= min_bucket
    merged["_cum_bad"] = merged.groupby(id_col)["_is_bad_event"].cummax()

    pivot = merged.pivot_table(
        index="vintage", columns=mob_col, values="_cum_bad", aggfunc="mean"
    ) * 100
    pivot = pivot.loc[:, pivot.columns <= max_mob]
    return pivot.round(2)


def plot_vintage_curves(pivot, title="Vintage Cumulative Bad Rate", save_path=None):
    fig, ax = plt.subplots(figsize=(10, 6))
    for vintage in pivot.index:
        series = pivot.loc[vintage].dropna()
        ax.plot(series.index, series.values, marker="o", markersize=3, label=vintage)
    ax.set_xlabel("Months on Book (MOB)")
    ax.set_ylabel("Cumulative Bad Rate (%)")
    ax.set_title(title)
    ax.legend(title="Vintage", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
    return fig


def find_maturity_point(pivot, marginal_threshold=0.3):
    """
    找到跨 Vintage 平均后的曲线开始"走平"的 MOB：
    当月度边际增量(Delta 累计Bad率) 连续低于 marginal_threshold(百分点)时，
    认为已接近成熟拐点 —— 可作为 Performance Window 长度的参考依据。
    """
    avg_curve = pivot.mean(axis=0, skipna=True).sort_index()
    marginal = avg_curve.diff()
    candidates = marginal[(marginal.index > 1) & (marginal.abs() < marginal_threshold)]
    mature_mob = candidates.index.min() if len(candidates) else np.nan
    return mature_mob, avg_curve, marginal


# =============================================================================
# 4. 一键跑完整流程
# =============================================================================

def run_full_analysis(df, id_col="CREDITCARDNO", snap_col="SNAPSHOT_DATE",
                       bucket_col="DELINQUENCYBUCKET", approve_col="APPROVEDATE",
                       bad_definitions=None, output_dir="."):
    df = prepare_data(df, id_col, snap_col, bucket_col, approve_col)

    print("\n========== 1. Roll Rate 分析 ==========")
    roll_rate_1m, _ = build_roll_rate_matrix(df, id_col, snap_col, bucket_col, n_periods=1)
    print("1个月滚动迁移矩阵 (%):")
    print(roll_rate_1m)
    plot_roll_rate_heatmap(roll_rate_1m, n_periods=1,
                            save_path=f"{output_dir}/roll_rate_1m.png")

    rate_30_to_90_6m, n_obs = eventual_roll_forward_rate(
        df, id_col, snap_col, bucket_col, from_bucket=2, to_bucket_min=4, within_months=6
    )
    print(f"\n一旦进入 30-59dpd(bucket=2)，未来6个月内滚到 90+dpd(bucket=4) 的概率: "
          f"{rate_30_to_90_6m:.1f}%  (基于 {n_obs} 个观察点)")

    print("\n========== 2. 候选 Bad 定义标注与对比 ==========")
    account_summary, bad_definitions = define_bad_flags(
        df, id_col, bucket_col, "MOB", bad_definitions
    )
    summary_df, overlap_df = compare_bad_definitions(account_summary, bad_definitions)
    print("\n各 Bad 定义在成熟样本口径下的表现:")
    print(summary_df.to_string(index=False))
    print("\n定义两两对比 (一致性 & 相关性):")
    print(overlap_df.to_string(index=False))

    print("\n========== 3. Vintage 分析 ==========")
    # 以 12个月窗口对应的 bad 定义 (min_bucket=3, 即 60dpd) 为例画 vintage 曲线
    example_def_name = next(iter(bad_definitions))
    min_bucket_example = bad_definitions[example_def_name]["min_bucket"]
    pivot = vintage_cumulative_bad_rate(
        df, account_summary, id_col, bucket_col, min_bucket=min_bucket_example, max_mob=18
    )
    plot_vintage_curves(
        pivot, title=f"Vintage Cumulative Bad Rate (bucket>={min_bucket_example})",
        save_path=f"{output_dir}/vintage_curves.png"
    )
    mature_mob, avg_curve, marginal = find_maturity_point(pivot)
    print(f"\n跨 Vintage 平均累计坏账率曲线开始走平的 MOB (成熟点参考): {mature_mob}")
    print("\n平均累计坏账率曲线 (按 MOB):")
    print(avg_curve.round(2).to_string())

    return {
        "df_prepared": df,
        "roll_rate_1m": roll_rate_1m,
        "account_summary": account_summary,
        "bad_definition_summary": summary_df,
        "bad_definition_overlap": overlap_df,
        "vintage_pivot": pivot,
        "maturity_mob": mature_mob,
    }


# =============================================================================
# 示例：用模拟数据跑通全流程（替换成你自己的真实数据即可）
# =============================================================================

# if __name__ == "__main__":
#     rng = np.random.RandomState(42)

#     n_accounts = 800
#     approve_dates = pd.to_datetime(
#         rng.choice(pd.date_range("2024-01-01", "2025-06-01", freq="MS"), n_accounts)
#     )
#     snapshot_end = pd.Timestamp("2026-08-01")

#     records = []
#     # 简单的马尔可夫链模拟逐月 bucket 演化: 0->1->2->3->4, 也允许部分回正常
#     trans_matrix = {
#         0: [0.90, 0.10, 0.00, 0.00, 0.00],
#         1: [0.35, 0.35, 0.30, 0.00, 0.00],
#         2: [0.10, 0.20, 0.30, 0.40, 0.00],
#         3: [0.05, 0.05, 0.10, 0.30, 0.50],
#         4: [0.02, 0.02, 0.02, 0.04, 0.90],
#     }

#     for i in range(n_accounts):
#         acc_id = f"CC{i:06d}"
#         act_date = approve_dates[i]
#         month = act_date
#         bucket = 0
#         while month <= snapshot_end:
#             records.append((acc_id, month, bucket, act_date))
#             probs = trans_matrix[bucket]
#             bucket = rng.choice([0, 1, 2, 3, 4], p=probs)
#             month = month + pd.DateOffset(months=1)

#     raw_df = pd.DataFrame(
#         records, columns=["CREDITCARDNO", "SNAPSHOT_DATE", "DELINQUENCYBUCKET", "APPROVEDATE"]
#     )

#     bad_defs = {
#         "Ever60DPD_12M": {"min_bucket": 3, "window_months": 12},
#         "Ever30DPD_6M":  {"min_bucket": 2, "window_months": 6},
#         "Ever90DPD_12M": {"min_bucket": 4, "window_months": 12},
#     }

#     results = run_full_analysis(raw_df, bad_definitions=bad_defs, output_dir="/home/claude")
